In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df= pd.read_csv("adult_with_headers (1).csv" )

In [13]:
df.shape

(32561, 15)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [15]:
df = df.replace( " ?", np.nan )
for col in ["workclass", "occupation", "native_country"]:
    df[col] = df[col].fillna(df[col].mode()[0])
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

We replaced all ' ?' values with NaN. Then we filled them with the mode (most common value) of each column.

In [16]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
num_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
ss = StandardScaler( )
df_std = pd.DataFrame( ss.fit_transform( df[num_cols] ), columns=num_cols )
mm = MinMaxScaler( )
df_mm = pd.DataFrame( mm.fit_transform( df[num_cols] ), columns=num_cols )
print(df_std.head(2))
print(df_mm.head(2))

        age    fnlwgt  education_num  capital_gain  capital_loss  \
0  0.030671 -1.063611       1.134739      0.148453      -0.21666   
1  0.837109 -1.008707       1.134739     -0.145920      -0.21666   

   hours_per_week  
0       -0.035429  
1       -2.222153  
        age    fnlwgt  education_num  capital_gain  capital_loss  \
0  0.301370  0.044302            0.8       0.02174           0.0   
1  0.452055  0.048238            0.8       0.00000           0.0   

   hours_per_week  
0        0.397959  
1        0.122449  


Standard scaling is used to center variables to mean 0 and variance 1, which helps algorithms like logistic regression. Min-max scaling squashes values between 0 and 1, which is good for distance calculations like K-Nearest Neighbors.

In [17]:
from sklearn.preprocessing import LabelEncoder
df = pd.get_dummies(df, columns=["sex", "income"], drop_first=True)
le = LabelEncoder()
large_cats = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "native_country"]
for col in large_cats:
    df[col] = le.fit_transform(df[col])
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,capital_gain,capital_loss,hours_per_week,native_country,sex_ Male,income_ >50K
0,39,6,77516,9,13,4,0,1,4,2174,0,40,38,True,False
1,50,5,83311,9,13,2,3,0,4,0,0,13,38,True,False
2,38,3,215646,11,9,0,5,1,4,0,0,40,38,True,False
3,53,3,234721,1,7,2,5,0,2,0,0,40,38,True,False
4,28,3,338409,9,13,2,9,5,2,0,0,40,4,False,False


One-hot encoding is used for columns with few categories (under 5) like sex and income to prevent artificial ordering. Label encoding is used for columns with many categories to keep the dataset small and prevent column explosion.

In [18]:
df["net_gain"] = df["capital_gain"] - df["capital_loss"]
df["age_hours"] = df["age"] * df["hours_per_week"]
df["log_capital_gain"] = np.log1p(df["capital_gain"])
df[["net_gain", "age_hours", "log_capital_gain"]].head()

,net_gain,age_hours,log_capital_gain
0,2174,1560,7.684784
1,0,650,0.000000
2,0,1520,0.000000
3,0,2120,0.000000
4,0,1120,0.000000


We created net_gain to show overall capital profit and age_hours to see working effort as age increases. We log-transformed capital_gain because it has extreme values that skew the model.

In [19]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest( contamination=0.05, random_state=42 )
df["outlier"] = iso.fit_predict( df[num_cols] )
print(df["outlier"].value_counts())

outlier
 1    30933
-1     1628
Name: count, dtype: int64


In [20]:
import ppscore as pps
df_sample = df.sample( 1000, random_state=42 )
matrix = pps.matrix( df_sample )
print(matrix.head(10))

     x               y   ppscore            case  is_valid_score  \
0  age             age  1.000000  predict_itself            True   
1  age       workclass  0.000000      regression            True   
2  age          fnlwgt  0.000000      regression            True   
3  age       education  0.000000      regression            True   
4  age   education_num  0.000000      regression            True   
5  age  marital_status  0.132289      regression            True   
6  age      occupation  0.000000      regression            True   
7  age    relationship  0.000000      regression            True   
8  age            race  0.000000      regression            True   
9  age    capital_gain  0.000000      regression            True   

                metric  baseline_score   model_score                    model  
0                 None           0.000      1.000000                     None  
1  mean absolute error           0.492      0.652571  DecisionTreeRegressor()  
2  mean abs